# ZeroDefect: Automated Optical Surface Inspection (Google Colab)

End-to-end surface defect inspection on the real steel-surface dataset with pixel-level ground-truth masks.

**Pipeline:** preprocessing → edge detection → thresholding → morphology → region analysis → evaluation (IoU, F1, precision, recall).

Run the cells top to bottom (**Runtime → Run all**). No GPU needed.

## 1. Setup: get the project and dependencies

In [ ]:
import os, sys

REPO_URL = "https://github.com/nikitatripathi10/zero-defect.git"
REPO_DIR = "zero-defect"

# Clone once (Colab sessions start empty)
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL}
%cd {REPO_DIR}

# Colab already ships numpy, pandas, matplotlib, scikit-learn, scikit-image, scipy
!pip install -q opencv-python-headless seaborn

sys.path.insert(0, os.getcwd())
print("Project root:", os.getcwd())

## 2. Load the real dataset

In [ ]:
import glob
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt

SURF_DIR = "data/surfaces"
GT_DIR   = "data/ground_truth"
CLASSES  = sorted(d for d in os.listdir(SURF_DIR) if os.path.isdir(os.path.join(SURF_DIR, d)))

rows = [(c, len(glob.glob(f"{SURF_DIR}/{c}/*"))) for c in CLASSES]
print(pd.DataFrame(rows, columns=["defect class", "images"]).to_string(index=False))
print("Total images:", sum(n for _, n in rows))

# One sample per class
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, c in zip(axes.ravel(), CLASSES):
    p = sorted(glob.glob(f"{SURF_DIR}/{c}/*"))[0]
    ax.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)); ax.set_title(c); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Inspect one image (preprocessing)

In [ ]:
from src.preprocessing import load_image, preprocess

DEFECT_CLASS = "scratches"   # change to: crazing, inclusion, patches, pitted_surface, rolled-in_scale
img_path = sorted(glob.glob(f"{SURF_DIR}/{DEFECT_CLASS}/*"))[0]
name = os.path.splitext(os.path.basename(img_path))[0]
image = load_image(img_path)
prep = preprocess(image, blur_method="gaussian", blur_kernel=5, use_clahe=True, clahe_clip=2.0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (k, t) in zip(axes, [("original","Original"),("gray","Grayscale"),("blurred","Blurred"),("enhanced","CLAHE enhanced")]):
    ax.imshow(prep[k], cmap=None if k == "original" else "gray"); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()
print("Inspecting:", img_path)

## 4. Edge detection and thresholding

In [ ]:
from src.edge_detection import auto_canny
from src.segmentation import otsu_threshold, adaptive_threshold

enhanced = prep["enhanced"]
edges = auto_canny(enhanced)
_, otsu_mask = otsu_threshold(enhanced)
adapt_mask = adaptive_threshold(enhanced, block_size=31, C=5)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, im, t in zip(axes, [edges, otsu_mask, adapt_mask], ["Canny edges", "Otsu threshold", "Adaptive threshold"]):
    ax.imshow(im, cmap="gray"); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Texture analysis

In [ ]:
from src.texture_features import compute_glcm_features, compute_statistics, local_std_map

std_map = local_std_map(enhanced, window_size=15)
print("GLCM:", {k: round(float(v), 4) for k, v in compute_glcm_features(enhanced).items()})
print("Stats:", {k: round(float(v), 2) for k, v in compute_statistics(enhanced).items()})

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(enhanced, cmap="gray"); axes[0].set_title("Enhanced"); axes[0].axis("off")
axes[1].imshow(std_map, cmap="magma"); axes[1].set_title("Local std-dev map (anomaly signal)"); axes[1].axis("off")
plt.tight_layout(); plt.show()

## 6. Full detection and region analysis

In [ ]:
from src.defect_detection import detect_defects_classical

result = detect_defects_classical(image, min_defect_area=80)
print("Verdict      :", result["verdict"])
print("Defect count :", result["defect_count"])
print("Area flagged : %.2f%%" % result["flagged_pct"])

if result["regions"]:
    display(pd.DataFrame(result["regions"]).head(10))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image); axes[0].set_title("Original")
axes[1].imshow(result["clean_mask"], cmap="gray"); axes[1].set_title("Predicted defect mask")
axes[2].imshow(result["annotated"]); axes[2].set_title("Detected regions")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## 7. Evaluate against the ground-truth mask
White pixels in the mask = defective, black = clean surface.

In [ ]:
from src.evaluation import pixel_metrics

def load_gt(cls, stem, shape):
    p = f"{GT_DIR}/{cls}/{stem}.png"
    m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if m is None:
        return None
    if m.shape != shape:
        m = cv2.resize(m, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
    return m

gt = load_gt(DEFECT_CLASS, name, result["clean_mask"].shape)
m = pixel_metrics(result["clean_mask"], gt)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items()})

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].imshow(result["clean_mask"], cmap="gray"); axes[0].set_title("Predicted mask")
axes[1].imshow(gt, cmap="gray"); axes[1].set_title("Ground-truth mask")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## 8. Batch evaluation across the whole dataset

In [ ]:
records = []
for cls in CLASSES:
    for p in sorted(glob.glob(f"{SURF_DIR}/{cls}/*")):
        stem = os.path.splitext(os.path.basename(p))[0]
        im = load_image(p)
        res = detect_defects_classical(im, min_defect_area=80)
        g = load_gt(cls, stem, res["clean_mask"].shape)
        if g is None:
            continue
        mm = pixel_metrics(res["clean_mask"], g)
        records.append({"class": cls, "image": stem, "regions": res["defect_count"],
                        "precision": mm["precision"], "recall": mm["recall"],
                        "f1": mm["f1"], "iou": mm["iou"]})

df = pd.DataFrame(records)
summary = df.groupby("class")[["precision", "recall", "f1", "iou"]].mean().round(3)
display(summary)
print("Overall mean:", df[["precision", "recall", "f1", "iou"]].mean().round(3).to_dict())

In [ ]:
summary.plot(kind="bar", figsize=(11, 4.5), rot=20)
plt.title("Pixel-level metrics by defect class"); plt.ylabel("score"); plt.ylim(0, 1)
plt.legend(loc="upper right"); plt.tight_layout(); plt.show()

## 9. Export results

In [ ]:
os.makedirs("outputs", exist_ok=True)
df.to_csv("outputs/zerodefect_metrics.csv", index=False)
cv2.imwrite("outputs/predicted_mask.png", result["clean_mask"])
print("Saved to", os.path.abspath("outputs"))

# Optional: download from Colab
try:
    from google.colab import files
    files.download("outputs/zerodefect_metrics.csv")
except ImportError:
    pass

## 10. Try your own image (optional)
Upload a surface image and run the same pipeline.

In [ ]:
try:
    from google.colab import files
    up = files.upload()
    for fn in up:
        im = load_image(fn)
        r = detect_defects_classical(im, min_defect_area=80)
        print(fn, "->", r["verdict"], "| regions:", r["defect_count"], "| flagged: %.2f%%" % r["flagged_pct"])
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].imshow(im); ax[0].axis("off"); ax[1].imshow(r["annotated"]); ax[1].axis("off")
        plt.show()
except ImportError:
    print("File upload is available only on Google Colab.")